# Module 1: Text Similarity Search

This module uses OpenAI Embedding model and TiDB Serverless Vector Search.

## Install Dependencies

In [1]:
%pip install -q \
    openai==1.72.0 \
    peewee==3.17.9 \
    pymysql==1.1.1 \
    tidb_vector==0.0.14

## Preapre the environment

> **Note:**
>
> - You can get the `OPENAI_API_KEY` from [OpenAI](https://platform.openai.com/docs/quickstart).
> - We already set the `TIDB_HOST`, `TIDB_PORT`, `TIDB_USERNAME`, `TIDB_PASSWORD`, and `TIDB_DATABASE` in the environment parameters. If you want to use this code snippet out of TiDB Labs platform, please set them beforehand.

Set the embedding model as `text-embedding-3-small`, and
the amount of embedding dimensions is `1536`.

In [5]:
import getpass

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

embedding_model = "text-embedding-3-small"
embedding_dimensions = 1536

## Initial the Clients of OpenAI and Database

In [7]:
import os

from openai import OpenAI
from peewee import Model, MySQLDatabase, TextField, SQL
from tidb_vector.peewee import VectorField

client = OpenAI(api_key=OPENAI_API_KEY)
db = MySQLDatabase(
    os.getenv("TIDB_DATABASE"),
    user=os.getenv("TIDB_USERNAME"),
    password=os.getenv("TIDB_PASSWORD"),
    host=os.getenv("TIDB_HOST"),
    port=int(os.getenv("TIDB_PORT")),
    ssl_verify_cert=True,
    ssl_verify_identity=True
)
db.connect()

## Prepare the Context

In this case, contexts are the documents, use the openai embeddings model to get the embeddings of the documents, and store them in the TiDB.

In [8]:
documents = [
   "TiDB is an open-source distributed SQL database that supports Hybrid Transactional and Analytical Processing (HTAP) workloads.",
   "TiFlash is the key component that makes TiDB essentially an Hybrid Transactional/Analytical Processing (HTAP) database. As a columnar storage extension of TiKV, TiFlash provides both good isolation level and strong consistency guarantee.",
   "TiKV is a distributed and transactional key-value database, which provides transactional APIs with ACID compliance. With the implementation of the Raft consensus algorithm and consensus state stored in RocksDB, TiKV guarantees data consistency between multiple replicas and high availability. ",
]

class DocModel(Model):
    text = TextField()
    embedding = VectorField(dimensions=embedding_dimensions)

    class Meta:
        database = db
        table_name = "openai_embedding_test"

    def __str__(self):
        return self.text

db.drop_tables([DocModel])
db.create_tables([DocModel])

embeddings = [
    r.embedding
    for r in client.embeddings.create(
      input=documents, model=embedding_model
    ).data
]
data_source = [
    {"text": doc, "embedding": emb}
    for doc, emb in zip(documents, embeddings)
]
DocModel.insert_many(data_source).execute()

## Initial the Vector of Question

Ask a question, use the openai embeddings model to get the embeddings of the question

In [9]:
question = "what is TiKV?"
question_embedding = client.embeddings.create(input=question, model=embedding_model).data[0].embedding

## Retrieve by Cosine Distance of Vectors
Get the relevant documents from the TiDB by comparing the embeddings of the question and the documents

In [10]:
related_docs = DocModel.select(
    DocModel.text, DocModel.embedding.cosine_distance(question_embedding).alias("distance")
).order_by(SQL("distance")).limit(3)

print("Question:", question)
print("Related documents:")
for doc in related_docs:
    print(doc.distance, doc.text)

## Cleanup

In [11]:
db.close()